# ROI Grad-CAM Calibration Notebook

This notebook replaces the one-off ROI export scripts with a configurable workflow for:

- loading a trained checkpoint,
- running Grad-CAM on **all images from train + validation**, 
- comparing multiple ROI thresholds side by side,
- visualizing many examples so you can calibrate the best ROI setting,
- exporting the selected ROI records as JSON in the same format used by training.

Recommended workflow:

1. Edit only the configuration cell.
2. Run the notebook top to bottom.
3. Inspect the threshold summary tables.
4. Inspect the multi-image galleries and the single-image threshold sweep.
5. Adjust thresholds and rerun the analysis / export cells until the ROIs look right.


In [ ]:
import inspect
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import warnings
from IPython.display import display
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

try:
    import ipywidgets as widgets
    from ipywidgets import fixed
    HAS_IPYWIDGETS = True
except ImportError:
    widgets = None
    fixed = None
    HAS_IPYWIDGETS = False
    warnings.warn(
        'ipywidgets is not installed in this environment. The notebook will still work, but the interactive sliders will be unavailable.'
    )

from data import SimpleDataset, build_dataset_dataframe, build_eval_transform, build_train_val_dataframes
from gradcam import compute_vit_gradcam_batch
from model import Model, load_model_checkpoint, resolve_model_kwargs_from_checkpoint
from roi_guidance import build_roi_record_from_cam, crop_image_to_roi, save_roi_records_to_json

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = False

# ======================
# USER CONFIGURATION
# ======================
CHECKPOINT_PATH = './checkpoints/P2_Temp_0.1_t1_finetune_best.pt'
DATA_DIR = '../data/Challenge_train_data'
OUTPUT_JSON_PATH = './checkpoints/roi_records/rois.json'

TARGET_CLASS = 1
EXPORT_LABEL_FILTER = 1  # Set to None to export ROIs for every image instead of only the positive class.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

FALLBACK_BACKBONE_NAME = 'vit_base_patch14_reg4_dinov2'
FALLBACK_INPUT_SIZE = 336

BATCH_SIZE = 8
NUM_WORKERS = 4

PROBABILITY_THRESHOLD = 0.50
THRESHOLDS_TO_COMPARE = [0.40, 0.50, 0.60, 0.70]
DEFAULT_EXPORT_THRESHOLD = 0.60

EXAMPLE_LABEL_FILTER = None  # None = all images, 1 = neo only, 0 = ndbe only.
EXAMPLE_MODE = 'near_prob_threshold'  # random | top_prob | low_prob | near_prob_threshold | high_cam_peak
EXAMPLE_COUNT = 6
EXAMPLE_RANDOM_SEED = 42
SINGLE_IMAGE_PATH = None  # Paste an absolute image path here for a focused threshold sweep.

OVERLAY_ALPHA = 0.80
ROI_CONTEXT_SCALE = 1.8
ROI_MIN_CROP_SCALE = 0.30
ROI_CENTER_JITTER = 0.00  # Keep 0.0 for deterministic preview crops.

print(f'Using device: {DEVICE}')


In [ ]:
def filter_model_kwargs_for_init(model_kwargs):
    valid_keys = {
        key for key in inspect.signature(Model.__init__).parameters
        if key not in {'self', 'kwargs'}
    }
    return {key: value for key, value in dict(model_kwargs).items() if key in valid_keys}


def load_all_train_val_dataframe(data_dir, split_random_state=42):
    full_df, class_names = build_dataset_dataframe(data_dir)
    train_df, val_df, split_class_names = build_train_val_dataframes(
        data_dir,
        random_state=split_random_state,
    )

    if class_names != split_class_names:
        raise ValueError('Class names from the full dataframe and split dataframe do not match.')

    train_df = train_df.copy()
    val_df = val_df.copy()
    train_df['split'] = 'train'
    val_df['split'] = 'val'

    split_df = pd.concat([train_df, val_df], ignore_index=True)
    split_df['img'] = split_df['img'].astype(str)

    full_df = full_df.copy()
    full_df['img'] = full_df['img'].astype(str)

    if len(full_df) != len(split_df):
        raise ValueError('Train + val does not cover the same number of images as the full dataframe.')
    if set(full_df['img']) != set(split_df['img']):
        raise ValueError('Train + val does not cover exactly the same image paths as the full dataframe.')

    split_map = split_df.set_index('img')['split'].to_dict()
    full_df['split'] = full_df['img'].map(split_map).fillna('unknown')
    return full_df.reset_index(drop=True), class_names, train_df.reset_index(drop=True), val_df.reset_index(drop=True)


def load_roi_source_model(checkpoint_path, class_names, device, fallback_backbone_name, fallback_input_size):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')

    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    resolved_model_kwargs = resolve_model_kwargs_from_checkpoint(
        checkpoint,
        fallback_kwargs={
            'in_channels': 3,
            'n_classes': len(class_names),
            'backbone_name': fallback_backbone_name,
            'input_size': fallback_input_size,
            'pretrained': False,
        },
    )
    resolved_model_kwargs = filter_model_kwargs_for_init(resolved_model_kwargs)
    resolved_model_kwargs['n_classes'] = len(class_names)

    model = Model(**resolved_model_kwargs).to(device)
    load_model_checkpoint(model, checkpoint_path, map_location=device)
    model.eval()
    return model, resolved_model_kwargs, checkpoint


def compute_gradcam_catalog(df, model, input_size, target_class, batch_size, num_workers, device):
    eval_df = df.copy().reset_index(drop=True)
    eval_ds = SimpleDataset(eval_df[['img', 'label']].copy(), build_eval_transform(input_size))
    eval_loader = DataLoader(
        eval_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device.type == 'cuda'),
    )

    records = []
    cam_store = {}
    path_offset = 0
    was_training = model.training
    model.eval()

    try:
        for images, labels in tqdm(eval_loader, desc='Computing Grad-CAMs'):
            batch_rows = eval_df.iloc[path_offset:path_offset + len(labels)]
            path_offset += len(labels)

            images = images.to(device)
            cams, probs = compute_vit_gradcam_batch(
                model=model,
                images=images,
                target_class=target_class,
            )
            cams = cams.detach().cpu()
            probs = probs.detach().cpu().numpy()

            for row, cam_tensor, prob in zip(batch_rows.itertuples(index=False), cams, probs):
                image_path = str(row.img)
                cam_store[image_path] = cam_tensor.numpy().astype(np.float16)
                records.append({
                    'img': image_path,
                    'label': int(row.label),
                    'center': str(row.center),
                    'split': str(row.split),
                    'positive_prob': float(prob),
                    'cam_peak': float(cam_tensor.max().item()),
                    'cam_mean': float(cam_tensor.mean().item()),
                })
    finally:
        if was_training:
            model.train()

    results_df = pd.DataFrame(records).reset_index(drop=True)
    return results_df, cam_store


In [ ]:
all_df, class_names, train_df, val_df = load_all_train_val_dataframe(DATA_DIR)
label_names = {idx: name for idx, name in enumerate(class_names)}

all_df['label_name'] = all_df['label'].map(label_names)
train_df['label_name'] = train_df['label'].map(label_names)
val_df['label_name'] = val_df['label'].map(label_names)

model, resolved_model_kwargs, checkpoint = load_roi_source_model(
    checkpoint_path=CHECKPOINT_PATH,
    class_names=class_names,
    device=DEVICE,
    fallback_backbone_name=FALLBACK_BACKBONE_NAME,
    fallback_input_size=FALLBACK_INPUT_SIZE,
)
INPUT_SIZE = int(resolved_model_kwargs.get('input_size', FALLBACK_INPUT_SIZE))

print(f'Loaded checkpoint: {Path(CHECKPOINT_PATH).resolve()}')
print(f'Backbone: {resolved_model_kwargs.get("backbone_name")}')
print(f'Input size: {INPUT_SIZE}')
print(f'Class names: {class_names}')
print(f'Total images used for analysis (train + val): {len(all_df)}')

display(pd.DataFrame({
    'subset': ['train', 'val', 'train+val'],
    'count': [len(train_df), len(val_df), len(all_df)],
}))

display(
    all_df.groupby(['split', 'label_name'])
    .size()
    .rename('count')
    .reset_index()
    .sort_values(['split', 'label_name'])
)

display(all_df.head())


In [ ]:
results_df, cam_store = compute_gradcam_catalog(
    df=all_df,
    model=model,
    input_size=INPUT_SIZE,
    target_class=TARGET_CLASS,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    device=DEVICE,
)

results_df['label_name'] = results_df['label'].map(label_names)
results_df = results_df.sort_values(['positive_prob', 'cam_peak'], ascending=[False, False]).reset_index(drop=True)

print(f'Computed Grad-CAMs for {len(results_df)} images.')

display(
    results_df.groupby('label_name')['positive_prob']
    .describe()
    .round(4)
)

display(results_df.head(10))


In [ ]:
def load_image_rgb(image_path):
    return np.asarray(Image.open(image_path).convert('RGB'))


def resize_cam_to_image(cam, image_rgb):
    cam = np.asarray(cam, dtype=np.float32)
    image_h, image_w = image_rgb.shape[:2]
    if cam.shape == (image_h, image_w):
        return cam
    cam_uint8 = np.clip(np.rint(cam * 255.0), 0, 255).astype(np.uint8)
    resized = Image.fromarray(cam_uint8, mode='L').resize((image_w, image_h), Image.BILINEAR)
    return np.asarray(resized, dtype=np.float32) / 255.0


def overlay_heatmap(image_rgb, cam, max_alpha=0.8):
    cam = resize_cam_to_image(cam, image_rgb)
    cam = np.clip(cam.astype(np.float32), 0.0, 1.0)
    heatmap_rgb = (plt.get_cmap('inferno')(cam)[..., :3] * 255.0).astype(np.uint8)
    alpha = cam[..., None] * float(max_alpha)
    overlay = (1.0 - alpha) * image_rgb.astype(np.float32) + alpha * heatmap_rgb.astype(np.float32)
    return np.clip(np.rint(overlay), 0, 255).astype(np.uint8)


def overlay_mask(image_rgb, mask, color=(255, 255, 0), alpha=0.35):
    base = image_rgb.astype(np.float32).copy()
    tint = np.asarray(color, dtype=np.float32).reshape(1, 1, 3)
    mask = resize_cam_to_image(mask.astype(np.float32), image_rgb) >= 0.5
    mask = mask.astype(bool)[..., None]
    blended = np.where(mask, (1.0 - alpha) * base + alpha * tint, base)
    return np.clip(np.rint(blended), 0, 255).astype(np.uint8)


def draw_bbox(image_rgb, bbox, color=(0, 255, 128), width=4):
    image = Image.fromarray(image_rgb.copy())
    draw = ImageDraw.Draw(image)
    img_w, img_h = image.size
    x0, y0, x1, y1 = [float(value) for value in bbox]
    left = int(round(x0 * img_w))
    top = int(round(y0 * img_h))
    right = int(round(x1 * img_w))
    bottom = int(round(y1 * img_h))

    for offset in range(width):
        draw.rectangle(
            [left - offset, top - offset, right + offset, bottom + offset],
            outline=tuple(color),
        )
    return np.asarray(image)


def build_roi_records_for_threshold(results_df, cam_store, threshold, prob_threshold, label_filter=None):
    candidate_df = results_df.copy()
    if label_filter is not None:
        candidate_df = candidate_df.loc[candidate_df['label'] == int(label_filter)].copy()

    passed_prob_df = candidate_df.loc[candidate_df['positive_prob'] >= float(prob_threshold)].copy()
    roi_records = {}
    coverage_values = []

    for row in passed_prob_df.itertuples(index=False):
        cam = cam_store[row.img].astype(np.float32)
        roi_record = build_roi_record_from_cam(
            cam,
            threshold=float(threshold),
            score=float(row.positive_prob),
        )
        if roi_record is None:
            continue
        roi_records[row.img] = roi_record
        coverage_values.append(float(roi_record['coverage']))

    summary = {
        'threshold': float(threshold),
        'prob_threshold': float(prob_threshold),
        'label_filter': label_filter,
        'candidate_images': int(len(candidate_df)),
        'passed_prob_threshold': int(len(passed_prob_df)),
        'roi_records_total': int(len(roi_records)),
        'acceptance_rate_vs_candidates': float(len(roi_records) / len(candidate_df)) if len(candidate_df) else 0.0,
        'acceptance_rate_vs_prob_filtered': float(len(roi_records) / len(passed_prob_df)) if len(passed_prob_df) else 0.0,
        'mean_coverage': float(np.mean(coverage_values)) if coverage_values else 0.0,
        'median_coverage': float(np.median(coverage_values)) if coverage_values else 0.0,
    }
    return roi_records, summary


def compare_thresholds(results_df, cam_store, thresholds, prob_threshold, label_filter=None):
    summaries = []
    for threshold in thresholds:
        _, summary = build_roi_records_for_threshold(
            results_df=results_df,
            cam_store=cam_store,
            threshold=threshold,
            prob_threshold=prob_threshold,
            label_filter=label_filter,
        )
        summaries.append(summary)
    return pd.DataFrame(summaries).sort_values('threshold').reset_index(drop=True)


def select_example_rows(results_df, count, label_filter=None, mode='random', prob_threshold=0.5, random_seed=42):
    subset = results_df.copy()
    if label_filter is not None:
        subset = subset.loc[subset['label'] == int(label_filter)].copy()
    if subset.empty:
        return subset.reset_index(drop=True)

    if mode == 'random':
        return subset.sample(n=min(count, len(subset)), random_state=random_seed).reset_index(drop=True)
    if mode == 'top_prob':
        return subset.sort_values('positive_prob', ascending=False).head(count).reset_index(drop=True)
    if mode == 'low_prob':
        return subset.sort_values('positive_prob', ascending=True).head(count).reset_index(drop=True)
    if mode == 'near_prob_threshold':
        subset = subset.assign(prob_margin=(subset['positive_prob'] - float(prob_threshold)).abs())
        return subset.sort_values(['prob_margin', 'cam_peak'], ascending=[True, False]).head(count).reset_index(drop=True)
    if mode == 'high_cam_peak':
        return subset.sort_values('cam_peak', ascending=False).head(count).reset_index(drop=True)

    raise ValueError(f'Unsupported EXAMPLE_MODE: {mode}')


def show_single_image_sweep(image_path, thresholds, prob_threshold, label_filter=None):
    image_path = str(image_path)
    row = results_df.loc[results_df['img'] == image_path]
    if row.empty:
        raise ValueError(f'Image path not found in results: {image_path}')
    row = row.iloc[0]

    image_rgb = load_image_rgb(image_path)
    cam = cam_store[image_path].astype(np.float32)
    overlay = overlay_heatmap(image_rgb, cam, max_alpha=OVERLAY_ALPHA)
    eligible = bool(row['positive_prob'] >= float(prob_threshold))
    if label_filter is not None:
        eligible = eligible and int(row['label']) == int(label_filter)

    thresholds = [float(value) for value in thresholds]
    ncols = 2 + len(thresholds)
    fig, axes = plt.subplots(2, ncols, figsize=(4 * ncols, 8))
    axes = np.asarray(axes)

    axes[0, 0].imshow(image_rgb)
    axes[0, 0].set_title(
        f'Original\nlabel={row["label_name"]} | prob={row["positive_prob"]:.3f}'
    )
    axes[0, 0].axis('off')

    axes[0, 1].imshow(overlay)
    axes[0, 1].set_title('Grad-CAM overlay')
    axes[0, 1].axis('off')

    axes[1, 0].imshow(cam, cmap='inferno', vmin=0.0, vmax=1.0)
    axes[1, 0].set_title('Normalized Grad-CAM')
    axes[1, 0].axis('off')

    axes[1, 1].axis('off')
    axes[1, 1].text(
        0.02,
        0.98,
        '\n'.join([
            f'Image: {Path(image_path).name}',
            f'Center: {row["center"]}',
            f'Split: {row["split"]}',
            f'Eligible for export: {eligible}',
            f'Prob threshold: {prob_threshold:.2f}',
            f'Export label filter: {label_filter}',
        ]),
        va='top',
        ha='left',
        fontsize=11,
        family='monospace',
    )

    for col_idx, threshold in enumerate(thresholds, start=2):
        mask = cam >= threshold
        panel = overlay_mask(image_rgb, mask)
        crop_panel = None
        roi_record = None

        if eligible:
            roi_record = build_roi_record_from_cam(
                cam,
                threshold=threshold,
                score=float(row['positive_prob']),
            )

        if roi_record is not None:
            panel = draw_bbox(panel, roi_record['bbox'])
            crop_panel = np.asarray(
                crop_image_to_roi(
                    image=Image.fromarray(image_rgb),
                    bbox=roi_record['bbox'],
                    context_scale=ROI_CONTEXT_SCALE,
                    min_crop_scale=ROI_MIN_CROP_SCALE,
                    jitter_xy=(0.0, 0.0),
                )
            )
        else:
            crop_panel = np.zeros_like(image_rgb)

        axes[0, col_idx].imshow(panel)
        axes[0, col_idx].set_title(
            f'th={threshold:.2f}\nroi={"yes" if roi_record is not None else "no"}'
        )
        axes[0, col_idx].axis('off')

        axes[1, col_idx].imshow(crop_panel)
        axes[1, col_idx].set_title('ROI crop' if roi_record is not None else 'No ROI crop')
        axes[1, col_idx].axis('off')

    plt.tight_layout()
    plt.show()


def show_example_grid(results_df, threshold, prob_threshold, count=6, selection_label_filter=None, eligibility_label_filter=None, mode='random', random_seed=42):
    sample_df = select_example_rows(
        results_df=results_df,
        count=count,
        label_filter=selection_label_filter,
        mode=mode,
        prob_threshold=prob_threshold,
        random_seed=random_seed,
    )
    if sample_df.empty:
        print('No images matched the current example settings.')
        return sample_df

    fig, axes = plt.subplots(len(sample_df), 4, figsize=(16, 4 * len(sample_df)))
    axes = np.atleast_2d(axes)

    for row_idx, row in enumerate(sample_df.itertuples(index=False)):
        image_rgb = load_image_rgb(row.img)
        cam = cam_store[row.img].astype(np.float32)
        overlay = overlay_heatmap(image_rgb, cam, max_alpha=OVERLAY_ALPHA)
        mask = cam >= float(threshold)

        eligible = bool(row.positive_prob >= float(prob_threshold))
        if eligibility_label_filter is not None:
            eligible = eligible and int(row.label) == int(eligibility_label_filter)

        roi_record = None
        bbox_panel = overlay_mask(image_rgb, mask)
        crop_panel = np.zeros_like(image_rgb)

        if eligible:
            roi_record = build_roi_record_from_cam(
                cam,
                threshold=float(threshold),
                score=float(row.positive_prob),
            )

        if roi_record is not None:
            bbox_panel = draw_bbox(bbox_panel, roi_record['bbox'])
            crop_panel = np.asarray(
                crop_image_to_roi(
                    image=Image.fromarray(image_rgb),
                    bbox=roi_record['bbox'],
                    context_scale=ROI_CONTEXT_SCALE,
                    min_crop_scale=ROI_MIN_CROP_SCALE,
                    jitter_xy=(0.0, 0.0),
                )
            )

        axes[row_idx, 0].imshow(image_rgb)
        axes[row_idx, 0].set_title(f'Original\n{Path(row.img).name}')
        axes[row_idx, 0].axis('off')

        axes[row_idx, 1].imshow(overlay)
        axes[row_idx, 1].set_title(f'Grad-CAM\nprob={row.positive_prob:.3f}')
        axes[row_idx, 1].axis('off')

        axes[row_idx, 2].imshow(bbox_panel)
        axes[row_idx, 2].set_title(
            f'th={threshold:.2f}\nroi={"yes" if roi_record is not None else "no"}'
        )
        axes[row_idx, 2].axis('off')

        axes[row_idx, 3].imshow(crop_panel)
        axes[row_idx, 3].set_title('ROI crop')
        axes[row_idx, 3].axis('off')

        axes[row_idx, 0].set_ylabel(
            f'{row.label_name}\n{row.center}\n{row.split}',
            rotation=0,
            labelpad=40,
            va='center',
        )

    plt.tight_layout()
    plt.show()
    return sample_df


In [ ]:
all_threshold_summary = compare_thresholds(
    results_df=results_df,
    cam_store=cam_store,
    thresholds=THRESHOLDS_TO_COMPARE,
    prob_threshold=PROBABILITY_THRESHOLD,
    label_filter=None,
)

display(all_threshold_summary.round(4))

if EXPORT_LABEL_FILTER is not None:
    export_threshold_summary = compare_thresholds(
        results_df=results_df,
        cam_store=cam_store,
        thresholds=THRESHOLDS_TO_COMPARE,
        prob_threshold=PROBABILITY_THRESHOLD,
        label_filter=EXPORT_LABEL_FILTER,
    )
    print('Export subset summary:')
    display(export_threshold_summary.round(4))


In [ ]:
gallery_df = show_example_grid(
    results_df=results_df,
    threshold=DEFAULT_EXPORT_THRESHOLD,
    prob_threshold=PROBABILITY_THRESHOLD,
    count=EXAMPLE_COUNT,
    selection_label_filter=EXAMPLE_LABEL_FILTER,
    eligibility_label_filter=EXPORT_LABEL_FILTER,
    mode=EXAMPLE_MODE,
    random_seed=EXAMPLE_RANDOM_SEED,
)

if SINGLE_IMAGE_PATH is None:
    focus_df = select_example_rows(
        results_df=results_df,
        count=1,
        label_filter=EXAMPLE_LABEL_FILTER,
        mode=EXAMPLE_MODE,
        prob_threshold=PROBABILITY_THRESHOLD,
        random_seed=EXAMPLE_RANDOM_SEED,
    )
    if focus_df.empty:
        print('No focus image available for the current settings.')
    else:
        show_single_image_sweep(
            image_path=focus_df.iloc[0]['img'],
            thresholds=THRESHOLDS_TO_COMPARE,
            prob_threshold=PROBABILITY_THRESHOLD,
            label_filter=EXPORT_LABEL_FILTER,
        )
else:
    show_single_image_sweep(
        image_path=SINGLE_IMAGE_PATH,
        thresholds=THRESHOLDS_TO_COMPARE,
        prob_threshold=PROBABILITY_THRESHOLD,
        label_filter=EXPORT_LABEL_FILTER,
    )


## Interactive Slider Tuning

Use this section to tune the Grad-CAM probability cutoff and ROI threshold live. You can switch between images, move the sliders, and immediately compare the resulting ROI overlay and crop.


In [ ]:
def build_interactive_image_options(results_df, selection_label_filter=None, max_images=250):
    subset = results_df.copy()
    if selection_label_filter is not None:
        subset = subset.loc[subset['label'] == int(selection_label_filter)].copy()

    subset = subset.sort_values(['positive_prob', 'cam_peak'], ascending=[False, False]).head(max_images)
    options = []
    for row in subset.itertuples(index=False):
        label_name = getattr(row, 'label_name', str(row.label))
        display_name = (
            f'{Path(row.img).name} | {label_name} | split={row.split} | '
            f'prob={row.positive_prob:.3f} | peak={row.cam_peak:.3f}'
        )
        options.append((display_name, row.img))
    return options


def interactive_roi_preview(image_path, prob_threshold, roi_threshold, eligibility_label_filter):
    show_single_image_sweep(
        image_path=image_path,
        thresholds=[roi_threshold],
        prob_threshold=prob_threshold,
        label_filter=eligibility_label_filter,
    )


if not HAS_IPYWIDGETS:
    print('ipywidgets is not available. Install it to use the interactive slider section.')
else:
    interactive_options = build_interactive_image_options(
        results_df=results_df,
        selection_label_filter=EXAMPLE_LABEL_FILTER,
        max_images=300,
    )

    if not interactive_options:
        print('No images are available for interactive tuning with the current filters.')
    else:
        option_values = {value for _label, value in interactive_options}
        default_image_value = (
            SINGLE_IMAGE_PATH
            if SINGLE_IMAGE_PATH is not None and SINGLE_IMAGE_PATH in option_values
            else interactive_options[0][1]
        )
        image_dropdown = widgets.Dropdown(
            options=interactive_options,
            value=default_image_value,
            description='Image:',
            layout=widgets.Layout(width='95%'),
            style={'description_width': 'initial'},
        )
        prob_slider = widgets.FloatSlider(
            value=float(PROBABILITY_THRESHOLD),
            min=0.0,
            max=1.0,
            step=0.01,
            description='Prob cutoff:',
            readout_format='.2f',
            continuous_update=False,
            layout=widgets.Layout(width='95%'),
            style={'description_width': 'initial'},
        )
        roi_slider = widgets.FloatSlider(
            value=float(DEFAULT_EXPORT_THRESHOLD),
            min=0.0,
            max=1.0,
            step=0.01,
            description='ROI threshold:',
            readout_format='.2f',
            continuous_update=False,
            layout=widgets.Layout(width='95%'),
            style={'description_width': 'initial'},
        )
        eligibility_dropdown = widgets.Dropdown(
            options=[('All labels', None), ('NDBE only (0)', 0), ('NEO only (1)', 1)],
            value=EXPORT_LABEL_FILTER,
            description='Export label rule:',
            layout=widgets.Layout(width='95%'),
            style={'description_width': 'initial'},
        )

        display(
            widgets.VBox([
                widgets.HTML('<b>Interactive ROI tuning</b><br>Pick an image and move the sliders to preview ROI acceptance, box placement, and crop size.'),
                image_dropdown,
                prob_slider,
                roi_slider,
                eligibility_dropdown,
            ])
        )

        interactive_output = widgets.interactive_output(
            interactive_roi_preview,
            {
                'image_path': image_dropdown,
                'prob_threshold': prob_slider,
                'roi_threshold': roi_slider,
                'eligibility_label_filter': eligibility_dropdown,
            },
        )
        display(interactive_output)


## Generate And Save `rois.json`

Run this cell after you are happy with the thresholds. It uses the current notebook configuration (`DATA_DIR`, `CHECKPOINT_PATH`, `OUTPUT_JSON_PATH`, `PROBABILITY_THRESHOLD`, `DEFAULT_EXPORT_THRESHOLD`) and writes the ROI JSON file directly.


In [ ]:
def generate_and_save_roi_json(
    results_df,
    cam_store,
    checkpoint_path,
    data_dir,
    output_json_path,
    target_class,
    export_label_filter,
    prob_threshold,
    roi_threshold,
    input_size,
):
    export_roi_records, export_summary = build_roi_records_for_threshold(
        results_df=results_df,
        cam_store=cam_store,
        threshold=roi_threshold,
        prob_threshold=prob_threshold,
        label_filter=export_label_filter,
    )

    metadata = {
        'checkpoint': str(Path(checkpoint_path).resolve()),
        'data_dir': str(Path(data_dir).resolve()),
        'split': 'train+val',
        'image_count_total': int(len(results_df)),
        'target_class': int(target_class),
        'export_label_filter': None if export_label_filter is None else int(export_label_filter),
        'probability_threshold': float(prob_threshold),
        'roi_gradcam_threshold': float(roi_threshold),
        'input_size': int(input_size),
    }

    save_roi_records_to_json(output_json_path, export_roi_records, metadata=metadata)
    return export_roi_records, export_summary, metadata


saved_roi_records, saved_roi_summary, saved_roi_metadata = generate_and_save_roi_json(
    results_df=results_df,
    cam_store=cam_store,
    checkpoint_path=CHECKPOINT_PATH,
    data_dir=DATA_DIR,
    output_json_path=OUTPUT_JSON_PATH,
    target_class=TARGET_CLASS,
    export_label_filter=EXPORT_LABEL_FILTER,
    prob_threshold=PROBABILITY_THRESHOLD,
    roi_threshold=DEFAULT_EXPORT_THRESHOLD,
    input_size=INPUT_SIZE,
)

print(f'Saved {len(saved_roi_records)} ROI records to {Path(OUTPUT_JSON_PATH).resolve()}')
display(pd.DataFrame([saved_roi_summary]).round(4))
display(pd.DataFrame([saved_roi_metadata]))
list(saved_roi_records.items())[:3]
